# Obesity Level Prediction Using Supervised Machine Learning

Problem type:Multiclass classification

Dataset: Estimation of Obesity Levels Based on Eating Habits and Physical Condition

Algorithms:Logistic Regression and Random Forest Classifier

Group:9

Logistic Regression by: Kumarasinghe LHSV MS26924182

Random Forest by :  IWMHD BANDARA MS26929514


# Objective
Predict the obesity-level category (`NObeyesdad`)



In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42

In [ ]:
# Load the dataset
DATA_PATH = "ObesityDataSet_raw_and_data_sinthetic.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\\nColumn names:")
print(df.columns.tolist())

In [ ]:

display(df.info())
print("\\nMissing values per column:")
display(df.isnull().sum())
print("\\nObesity-level class counts:")
display(df["NObeyesdad"].value_counts())

plt.figure(figsize=(10, 5))
sns.countplot(data=df, x="NObeyesdad", order=df["NObeyesdad"].value_counts().index)
plt.title("Distribution of Obesity-Level Classes")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
TARGET = "NObeyesdad"
X = df.drop(columns=[TARGET])
y = df[TARGET]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_preprocessor, numeric_features),
    ("categorical", categorical_preprocessor, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training instances:", X_train.shape[0])
print("Testing instances:", X_test.shape[0])
print("Number of input features:", X.shape[1])
print("Number of classes:", y.nunique())

In [ ]:
#Logistic Regression
logistic_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))
])

logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

print("Logistic Regression results")
print("Accuracy:", round(accuracy_score(y_test, logistic_predictions), 4))
print("Precision (macro):", round(precision_score(y_test, logistic_predictions, average="macro", zero_division=0), 4))
print("Recall (macro):", round(recall_score(y_test, logistic_predictions, average="macro", zero_division=0), 4))
print("F1-score (macro):", round(f1_score(y_test, logistic_predictions, average="macro", zero_division=0), 4))
print("\\nClassification report:")
print(classification_report(y_test, logistic_predictions, zero_division=0))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, logistic_predictions, xticks_rotation=45, cmap="Blues"
)
plt.title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.show()